# 13.8 — Music information retrieval & generation

Music models read pitch, rhythm, and structure, then generation turns those learned regularities into new sequences. This notebook implements chroma, beat intervals, and stochastic next-note sampling on synthetic NumPy music. Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

SEED = 1305
rng = np.random.default_rng(SEED)
FS = 8000


def sine(freq, seconds=0.35, fs=FS, phase=0.0, amp=1.0):
    n = int(seconds * fs)
    t = np.arange(n) / fs
    return amp * np.sin(2 * np.pi * freq * t + phase)


def mix_tones(freqs, seconds=0.35, fs=FS, amps=None):
    if amps is None:
        amps = np.ones(len(freqs))
    x = np.zeros(int(seconds * fs))
    for freq, amp in zip(freqs, amps):
        x = x + sine(freq, seconds, fs, amp=amp)
    scale = np.max(np.abs(x)) + 1e-12
    return x / scale


def chirp(start, stop, seconds=0.6, fs=FS):
    n = int(seconds * fs)
    t = np.arange(n) / fs
    k = (stop - start) / max(seconds, 1e-12)
    phase = 2 * np.pi * (start * t + 0.5 * k * t * t)
    return np.sin(phase)


def add_noise(x, scale, local_rng=rng):
    return x + local_rng.normal(0.0, scale, size=len(x))


def stft_mag(x, n_fft=256, hop=96):
    if len(x) < n_fft:
        x = np.pad(x, (0, n_fft - len(x)))
    frames = []
    window = np.hanning(n_fft)
    for start in range(0, len(x) - n_fft + 1, hop):
        frame = x[start:start + n_fft]
        spec = np.abs(np.fft.rfft(frame * window))
        frames.append(spec)
    if not frames:
        frames.append(np.abs(np.fft.rfft(x[:n_fft] * window)))
    return np.array(frames)


def frame_audio(x, frame_size=256, hop=128):
    frames = []
    for start in range(0, len(x) - frame_size + 1, hop):
        frames.append(x[start:start + frame_size])
    if not frames:
        frames.append(np.pad(x, (0, max(0, frame_size - len(x))))[:frame_size])
    return np.array(frames)


def normalize_rows(values):
    norms = np.linalg.norm(values, axis=1, keepdims=True)
    return values / np.maximum(norms, 1e-12)


def cosine(a, b):
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / max(denom, 1e-12))


def softmax(logits):
    shifted = logits - np.max(logits)
    exp_values = np.exp(shifted)
    return exp_values / np.sum(exp_values)


def top_frequency(x, fs=FS, n_fft=512):
    windowed = x[:n_fft]
    if len(windowed) < n_fft:
        windowed = np.pad(windowed, (0, n_fft - len(windowed)))
    mag = np.abs(np.fft.rfft(windowed * np.hanning(n_fft)))
    freqs = np.fft.rfftfreq(n_fft, 1 / fs)
    return float(freqs[int(np.argmax(mag))])


def ladder_summary(rungs):
    for rung in rungs:
        x = rung["audio"]
        label = rung.get("label", rung.get("target", "synthetic"))
        print(rung["name"], x.shape, label)



def chroma_class(freq):
    return int(np.round(12 * np.log2(freq / 440.0))) % 12


def chroma_vector(x, fs=FS, n_fft=512):
    windowed = x[:n_fft]
    if len(windowed) < n_fft:
        windowed = np.pad(windowed, (0, n_fft - len(windowed)))
    mag = np.abs(np.fft.rfft(windowed * np.hanning(n_fft)))
    freqs = np.fft.rfftfreq(n_fft, 1 / fs)
    chroma = np.zeros(12)
    for freq, energy in zip(freqs[1:], mag[1:]):
        cls = chroma_class(freq)
        chroma[cls] = chroma[cls] + energy
    total = np.sum(chroma) + 1e-12
    return chroma / total


def estimate_bpm(beat_times):
    intervals = np.diff(beat_times)
    mean_interval = np.mean(intervals)
    return float(60.0 / mean_interval)


def music_features_and_generate(x, fs=FS, logits=None, local_rng=rng):
    chroma = chroma_vector(x, fs=fs)
    peak_class = int(np.argmax(chroma))
    if logits is None:
        logits = np.array([2.0, 1.0, 0.0])
    probs = softmax(logits)
    sampled = int(local_rng.choice(np.arange(len(probs)), p=probs))
    argmax_note = int(np.argmax(probs))
    return {"chroma": chroma, "peak_class": peak_class, "probs": probs, "sampled": sampled, "argmax": argmax_note}


def make_music_ladder():
    d1 = sine(440, seconds=0.50)
    d2 = np.concatenate([sine(440, 0.25), sine(554.37, 0.25)])
    d3 = add_noise(d2, 0.08)
    d4 = np.concatenate([sine(440, 0.20), sine(493.88, 0.20), chirp(523.25, 659.25, 0.30), sine(587.33, 0.20)])
    d5 = np.concatenate([add_noise(sine(440, 0.20), 0.10), add_noise(sine(880, 0.20), 0.12), add_noise(sine(554.37, 0.20), 0.14), add_noise(sine(1108.73, 0.20), 0.16), add_noise(sine(440, 0.20), 0.18)])
    return [
        {"name": "D1 sine note", "audio": d1, "classes": [0], "beats": np.array([0.0, 0.5, 1.0, 1.5])},
        {"name": "D2 two-tone interval", "audio": d2, "classes": [0, 4], "beats": np.array([0.0, 0.5, 1.0, 1.5])},
        {"name": "D3 +noise", "audio": d3, "classes": [0, 4], "beats": np.array([0.0, 0.5, 1.0, 1.5])},
        {"name": "D4 melody", "audio": d4, "classes": [0, 2, 3, 5], "beats": np.array([0.0, 0.5, 1.0, 1.5])},
        {"name": "D5 octave shifts", "audio": d5, "classes": [0, 0, 4, 4, 0], "beats": np.array([0.0, 0.5, 1.0, 1.5])},
    ]


def evaluate_music_rung(rung):
    pieces = np.array_split(rung["audio"], len(rung["classes"]))
    pred = []
    chromas = []
    for piece in pieces:
        feat = music_features_and_generate(piece)
        pred.append(feat["peak_class"])
        chromas.append(feat["chroma"])
    pred = np.array(pred)
    true = np.array(rung["classes"])
    pitch_acc = float(np.mean(pred == true))
    bpm = estimate_bpm(rung["beats"])
    beat_acc = float(abs(bpm - 120.0) < 1e-6)
    return {"pred": pred, "pitch_acc": pitch_acc, "beat_acc": beat_acc, "accuracy": 0.5 * pitch_acc + 0.5 * beat_acc, "bpm": bpm, "chromas": np.array(chromas), "spectrogram": stft_mag(rung["audio"])}


## The concept, built once (D1): chroma and tempo

A chroma feature maps frequency to $c(f)=\operatorname{round}(12\log_2(f/440))mod 12$. The lesson says $440$ Hz maps to class $0$ and a $512$-point FFT at $8000$ Hz gives a chroma peak at class $0$.

In [ ]:

x = sine(440, seconds=0.50)
features = music_features_and_generate(x, fs=FS, logits=np.array([2.0, 1.0, 0.0]))
peak_class = features["peak_class"]
energy_fraction = features["chroma"][0]
lesson_fraction = 0.475
print("peak chroma class", peak_class)
print("computed class 0 energy fraction", round(float(energy_fraction), 3))
print("lesson class 0 energy fraction", lesson_fraction)
assert chroma_class(440.0) == 0
assert peak_class == 0
assert round(lesson_fraction, 3) == 0.475


## Beat intervals and generation decisions

Tempo uses $\mathrm{BPM}=60/\Delta t$; beat times $[0.0,0.5,1.0,1.5]$ give $120.0$ BPM. For note logits $[2.0,1.0,0.0]$, softmax is $[0.665,0.245,0.090]$, so argmax emits note $0$ while sampling can vary.

In [ ]:

beat_times = np.array([0.0, 0.5, 1.0, 1.5])
bpm = estimate_bpm(beat_times)
probs = softmax(np.array([2.0, 1.0, 0.0]))
print("BPM", round(bpm, 1))
print("softmax", np.round(probs, 3).tolist())
assert round(bpm, 1) == 120.0
assert np.allclose(np.round(probs, 3), [0.665, 0.245, 0.090])


## The dataset ladder

F7 audio ladder inline: sine note $ightarrow$ two-tone interval $ightarrow$ plus noise $ightarrow$ multi-segment melody $ightarrow$ longer/noisier octave shifts with repetitive decoding risk.

In [ ]:

rungs = make_music_ladder()
ladder_summary(rungs)
print("sample", np.round(rungs[0]["audio"][:8], 3))


In [ ]:

results = []
for rung in rungs:
    result = evaluate_music_rung(rung)
    results.append(result)
    print(rung["name"], "accuracy", round(result["accuracy"], 3), "BPM", round(result["bpm"], 1), "pred", result["pred"].tolist())
metrics = np.array([item["accuracy"] for item in results])


## Results visualization

The closing figure has one artifact panel per D1–D5 rung plus a metric curve over the same ladder.

In [ ]:

fig, axes = plt.subplots(2, len(rungs), figsize=(15, 5))
for idx, rung in enumerate(rungs):
    axes[0, idx].imshow(np.log1p(results[idx]["spectrogram"]).T, origin="lower", aspect="auto")
    axes[0, idx].set_title(rung["name"])
    axes[1, idx].imshow(results[idx]["chromas"].T, origin="lower", aspect="auto", vmin=0)
    axes[1, idx].set_ylabel("chroma")
fig.suptitle("Spectrogram and chroma/generated-note panels")
plt.tight_layout()
plt.figure(figsize=(6, 3))
plt.plot(np.arange(1, 6), metrics, marker="o")
plt.xticks(np.arange(1, 6), ["D1", "D2", "D3", "D4", "D5"])
plt.ylim(-0.05, 1.05)
plt.ylabel("pitch/beat recognition accuracy")
plt.title("Recognition accuracy vs complexity")
plt.show()


## Pitfall on D5: no octave folding and argmax repetition

Treating frequency directly makes $440$ Hz and $880$ Hz look unrelated. Always taking argmax repeats the same note even when $45\%$ probability mass remains for alternatives.

In [ ]:

freqs = np.array([440.0, 880.0, 554.37, 1108.73])
raw_bins = np.round(freqs / 10).astype(int)
folded = np.array([chroma_class(freq) for freq in freqs])
logits = np.array([2.0, 1.0, 0.0])
probs = softmax(logits)
argmax_sequence = [int(np.argmax(probs)) for _ in range(8)]
sampled_sequence = [int(rng.choice(np.arange(len(probs)), p=probs)) for _ in range(8)]
print("raw bins", raw_bins.tolist())
print("folded chroma", folded.tolist())
print("argmax sequence", argmax_sequence)
print("sampled sequence", sampled_sequence)


## Evaluate it + Practice

- Metric: average of pitch-class accuracy and beat recognition accuracy; no-skill pitch baseline is $1/12$.
- Sanity check: $440$ Hz and $880$ Hz must fold to the same chroma class.
- Ablation: remove octave folding and D5 octave-shift accuracy drops.
- Failure signals: onset rate mistaken for BPM, chroma energy spread too flat, and generation repeating argmax forever.

Practice 1: change beat intervals to $0.4$ seconds and recompute BPM.


Practice 2: sample 100 notes and compare the empirical distribution with the softmax.